# 01 – Explore FUELHH (actual generation by fuel type)

In [1]:
import requests
import pandas as pd

BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1"

## Cell 2 – Raw API response for a single day

In [2]:
url = f"{BASE_URL}/datasets/FUELHH/stream"
response = requests.get(
    url,
    params={
        "settlementDateFrom": "2025-01-15",
        "settlementDateTo": "2025-01-15",
        "fuelType": "WIND",
    },
)
print(response.status_code)
print(response.url)

raw = response.json()
print(type(raw), len(raw))
raw[0]

200
https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH/stream?settlementDateFrom=2025-01-15&settlementDateTo=2025-01-15&fuelType=WIND
<class 'list'> 48


{'dataset': 'FUELHH',
 'publishTime': '2025-01-16T00:00:00Z',
 'startTime': '2025-01-15T23:30:00Z',
 'settlementDate': '2025-01-15',
 'settlementPeriod': 48,
 'fuelType': 'WIND',
 'generation': 5715}

In [3]:
# Did the fuelType filter work? Which fields come back?
print({row["fuelType"] for row in raw})
print(sorted(raw[0].keys()))
print(raw[0]["startTime"], "->", raw[-1]["startTime"])

{'WIND'}
['dataset', 'fuelType', 'generation', 'publishTime', 'settlementDate', 'settlementPeriod', 'startTime']
2025-01-15T23:30:00Z -> 2025-01-15T00:00:00Z


### Cell 2 findings

1. **`startTime` format:** an ISO 8601 string such as `"2025-01-15T23:30:00Z"`. The trailing `Z` means it's in UTC.
2. **Other fields:** `dataset`, `publishTime`, `settlementDate` and `settlementPeriod` (1–48) also come back. So yes, `settlementPeriod` is one of them. The stream endpoint returns a plain **list** of records, not a dict with a `data` key, and the rows come **newest first**.
3. **Fuel-type filter:** it worked on the API side. The only `fuelType` in the response was `WIND`, and one day came back as exactly 48 rows.


## Cell 3 – `fetch_actuals`

In [4]:
def fetch_actuals(start, end, fuel_type="WIND"):
    """Half-hourly actual generation (MW) for one fuel type, for settlement dates start..end inclusive."""
    response = requests.get(
        f"{BASE_URL}/datasets/FUELHH/stream",
        params={
            "settlementDateFrom": start,
            "settlementDateTo": end,
            "fuelType": fuel_type,
        },
    )
    response.raise_for_status()

    df = pd.DataFrame(response.json())[["startTime", "generation"]]
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    return df.sort_values("startTime").reset_index(drop=True)

## Cell 4 – Sanity checks

Range: **2025-03-27 → 2025-04-02** (7 settlement days). This deliberately includes the clocks-forward day on **Sun 30 March 2025**.

**Expectation, written before running:** 7 × 48 = 336 rows, minus 2 because 30 March is a 23-hour settlement day with only 46 periods. That gives **334**.
Boundaries: settlement days run on UK local time, so the first period should be `2025-03-27 00:00 UTC` (still GMT). The last should be `2025-04-02 22:30 UTC`, because the 2 April settlement day ends at 00:00 BST, which is 23:00 UTC.

In [5]:
actuals = fetch_actuals("2025-03-27", "2025-04-02")

print("rows:", len(actuals), "(expected 334)")
print("min startTime:", actuals["startTime"].min())
print("max startTime:", actuals["startTime"].max())
print("duplicate startTime:", actuals["startTime"].duplicated().sum())

gaps = actuals["startTime"].diff().dropna()
print("gap sizes:", gaps.value_counts().to_dict())
print("all gaps 30 min:", (gaps == pd.Timedelta(minutes=30)).all())

print("null generation:", actuals["generation"].isna().sum())
print("negative generation:", (actuals["generation"] < 0).sum())
print(actuals.dtypes)

rows: 334 (expected 334)
min startTime: 2025-03-27 00:00:00+00:00
max startTime: 2025-04-02 22:30:00+00:00
duplicate startTime: 0
gap sizes: {Timedelta('0 days 00:30:00'): 333}
all gaps 30 min: True
null generation: 0
negative generation: 0
startTime     datetime64[us, UTC]
generation                  int64
dtype: object


## Cell 5 – Raw look at WINDFOR

`/datasets/WINDFOR/stream` takes only **`publishDateTimeFrom` / `publishDateTimeTo`**. These are **publish-time** filters, and the endpoint has no `startTime` filter. Like FUELHH, it returns a plain list.

In [6]:
response = requests.get(
    f"{BASE_URL}/datasets/WINDFOR/stream",
    params={
        "publishDateTimeFrom": "2025-03-26T00:00:00Z",
        "publishDateTimeTo": "2025-03-27T00:00:00Z",
    },
)
print(response.status_code)
print(response.url)

raw_fc = pd.DataFrame(response.json())
print(raw_fc.iloc[0].to_dict())

for col in ["publishTime", "startTime"]:
    raw_fc[col] = pd.to_datetime(raw_fc[col], utc=True)
raw_fc["horizon_h"] = (raw_fc["startTime"] - raw_fc["publishTime"]).dt.total_seconds() / 3600

print("unique publishTimes:", raw_fc["publishTime"].nunique())
raw_fc.groupby("publishTime").agg(
    n=("startTime", "size"),
    start_min=("startTime", "min"),
    start_max=("startTime", "max"),
    horizon_min_h=("horizon_h", "min"),
    horizon_max_h=("horizon_h", "max"),
)

200
https://data.elexon.co.uk/bmrs/api/v1/datasets/WINDFOR/stream?publishDateTimeFrom=2025-03-26T00%3A00%3A00Z&publishDateTimeTo=2025-03-27T00%3A00%3A00Z
{'dataset': 'WINDFOR', 'publishTime': '2025-03-26T22:30:00Z', 'startTime': '2025-03-25T21:00:00Z', 'generation': 3393}
unique publishTimes: 8


,n,start_min,start_max,horizon_min_h,horizon_max_h
publishTime,,,,,
2025-03-26 02:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-5.5,66.5
2025-03-26 04:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-7.5,64.5
2025-03-26 07:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-10.5,61.5
2025-03-26 09:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-12.5,59.5
2025-03-26 11:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-14.5,57.5
2025-03-26 15:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-18.5,53.5
2025-03-26 18:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-21.5,50.5
2025-03-26 22:30:00+00:00,73,2025-03-25 21:00:00+00:00,2025-03-28 21:00:00+00:00,-25.5,46.5


In [7]:
one_publish = raw_fc[raw_fc["publishTime"] == raw_fc["publishTime"].min()]
one_publish["startTime"].sort_values().diff().value_counts()

startTime
0 days 01:00:00    72
Name: count, dtype: int64

### Cell 5 findings

1. **Publishes per day:** 8, at **02:30, 04:30, 07:30, 09:30, 11:30, 15:30, 18:30 and 22:30 UTC** on 26 March 2025, which is still GMT.
2. **Horizon:** every publish on a given day covers the **same fixed block** of `startTime`, from 21:00 on the previous day to 21:00 two days later (73 hourly points). The block doesn't move with the publish hour, so the horizon depends on when the forecast was published:
   - the 02:30 publish runs from **−5.5 h to +66.5 h**
   - the 22:30 publish runs from **−25.5 h to +46.5 h**

   **Horizons can be negative.** Every publish includes "forecasts" for hours that had already started when it was published. The overall maximum horizon is **66.5 h**.
3. **Spacing:** `startTime` values are **hourly**, not half-hourly like FUELHH.


## Cell 6 – `fetch_forecasts`

In [8]:
def fetch_forecasts(publish_from, publish_to):
    """All WINDFOR forecasts published between publish_from and publish_to (UTC ISO strings)."""
    response = requests.get(
        f"{BASE_URL}/datasets/WINDFOR/stream",
        params={
            "publishDateTimeFrom": publish_from,
            "publishDateTimeTo": publish_to,
        },
    )
    response.raise_for_status()

    df = pd.DataFrame(response.json())[["publishTime", "startTime", "generation"]]
    for col in ["publishTime", "startTime"]:
        df[col] = pd.to_datetime(df[col], utc=True)
    return df.sort_values(["startTime", "publishTime"]).reset_index(drop=True)

In [9]:
# Delivery window = same as actuals: 2025-03-27 00:00 UTC .. 2025-04-02 22:30 UTC.
# The longest horizon in Cell 5 was 66.5 h, so a forecast for the first delivery hour
# (2025-03-27 00:00) can be published as early as 2025-03-27 00:00 - 66.5 h = 2025-03-24 01:30.
# Round down to the start of that day: publish_from = 2025-03-24 00:00 UTC.
# Nothing published after the last delivery period started can count as a forecast for it,
# so publish_to = 2025-04-02 23:00 UTC.
forecasts = fetch_forecasts("2025-03-24T00:00:00Z", "2025-04-02T23:00:00Z")
forecasts.head()

,publishTime,startTime,generation
0,2025-03-24 02:30:00+00:00,2025-03-23 21:00:00+00:00,3342
1,2025-03-24 04:30:00+00:00,2025-03-23 21:00:00+00:00,3342
2,2025-03-24 07:30:00+00:00,2025-03-23 21:00:00+00:00,3342
3,2025-03-24 09:30:00+00:00,2025-03-23 21:00:00+00:00,3342
4,2025-03-24 11:30:00+00:00,2025-03-23 21:00:00+00:00,3342


## Cell 7 – Checks

In [10]:
# Restrict to the delivery window we care about (fetch_forecasts returns whole blocks around it)
delivery = forecasts[
    (forecasts["startTime"] >= "2025-03-27 00:00Z") & (forecasts["startTime"] <= "2025-04-02 22:30Z")
].copy()

per_start = delivery.groupby("startTime").size()
print("startTimes:", len(per_start), "| forecasts per startTime:")
print(per_start.value_counts().sort_index().to_string())

print("\nduplicate (publishTime, startTime):", delivery.duplicated(["publishTime", "startTime"]).sum())

delivery["horizon_h"] = (delivery["startTime"] - delivery["publishTime"]).dt.total_seconds() / 3600
print("horizon min/max (h):", delivery["horizon_h"].min(), delivery["horizon_h"].max())
print("rows with horizon <= 0:", (delivery["horizon_h"] <= 0).sum(), "of", len(delivery))

print("\npublish clock times (UTC):", sorted(forecasts["publishTime"].dt.strftime("%H:%M").unique()))

startTimes: 167 | forecasts per startTime:
15      2
23     24
24    135
31      1
32      5

duplicate (publishTime, startTime): 0
horizon min/max (h): -27.5 66.5
rows with horizon <= 0: 844 of 4013

publish clock times (UTC): ['02:30', '03:30', '04:30', '05:30', '07:30', '08:30', '09:30', '10:30', '11:30', '12:30', '15:30', '16:30', '18:30', '19:30', '22:30', '23:30']


In [11]:
# Where do the 'odd' counts come from? Show the startTimes whose count isn't the most common one.
mode = per_start.mode()[0]
per_start[per_start != mode]

startTime
2025-03-27 21:00:00+00:00    32
2025-03-28 21:00:00+00:00    32
2025-03-29 21:00:00+00:00    32
2025-03-30 20:00:00+00:00    32
2025-03-31 20:00:00+00:00    32
2025-04-01 20:00:00+00:00    31
2025-04-01 21:00:00+00:00    23
2025-04-01 22:00:00+00:00    23
2025-04-01 23:00:00+00:00    23
2025-04-02 00:00:00+00:00    23
2025-04-02 01:00:00+00:00    23
2025-04-02 02:00:00+00:00    23
2025-04-02 03:00:00+00:00    23
2025-04-02 04:00:00+00:00    23
2025-04-02 05:00:00+00:00    23
2025-04-02 06:00:00+00:00    23
2025-04-02 07:00:00+00:00    23
2025-04-02 08:00:00+00:00    23
2025-04-02 09:00:00+00:00    23
2025-04-02 10:00:00+00:00    23
2025-04-02 11:00:00+00:00    23
2025-04-02 12:00:00+00:00    23
2025-04-02 13:00:00+00:00    23
2025-04-02 14:00:00+00:00    23
2025-04-02 15:00:00+00:00    23
2025-04-02 16:00:00+00:00    23
2025-04-02 17:00:00+00:00    23
2025-04-02 18:00:00+00:00    23
2025-04-02 19:00:00+00:00    23
2025-04-02 20:00:00+00:00    23
2025-04-02 21:00:00+00:00    1

### Cell 7 findings (observations only, not fixed yet)

- **No duplicates** on `(publishTime, startTime)`.
- **24 forecasts per startTime is the normal case:** 3 daily blocks × 8 publishes each. The exceptions:
  - **32** at the block boundary hour (21:00 UTC, or 20:00 UTC after the clocks change). That hour is both the last hour of one block and the first hour of the next, so it gets 8 extra forecasts.
  - **23 and 15** at the end of the window. This is only because `publish_to` stops before the later blocks are published.
- **Horizon runs from −27.5 h to +66.5 h**, and **844 of 4,013 rows have horizon ≤ 0**. Those rows were published *after* their delivery hour had started, so they aren't real forecasts.
- **Publish times move after the clocks change on 30 March.** They go from 02:30, 04:30, … 22:30 UTC to 03:30, 05:30, … 23:30 UTC, which is one hour *later* in UTC. The block boundary moves the other way, from 21:00 UTC to 20:00 UTC (21:00 local both times).
- FUELHH is half-hourly but WINDFOR is hourly. Joining the two will need a decision about this.


## Cell 8 – January 2026: every forecast for one delivery slot

In [12]:
# Delivery month = January 2026. Max horizon is 66.5 h (~2.8 days), so start publishes
# a few days before 1 Jan to cover the first delivery hours: publish_from = 2025-12-28.
jan_forecasts = fetch_forecasts("2025-12-28T00:00:00Z", "2026-02-01T00:00:00Z")
print(len(jan_forecasts), "rows,", jan_forecasts["publishTime"].nunique(), "publishes")

slot = pd.Timestamp("2026-01-15 18:00", tz="UTC")
one_slot = jan_forecasts[jan_forecasts["startTime"] == slot].copy()
one_slot["horizon_h"] = (one_slot["startTime"] - one_slot["publishTime"]).dt.total_seconds() / 3600
one_slot = one_slot.sort_values("publishTime").reset_index(drop=True)
one_slot

20440 rows, 280 publishes


,publishTime,startTime,generation,horizon_h
0,2026-01-13 02:30:00+00:00,2026-01-15 18:00:00+00:00,10619,63.500000
1,2026-01-13 04:30:00+00:00,2026-01-15 18:00:00+00:00,11391,61.500000
2,2026-01-13 07:30:00+00:00,2026-01-15 18:00:00+00:00,10470,58.500000
3,2026-01-13 09:30:00+00:00,2026-01-15 18:00:00+00:00,10005,56.500000
4,2026-01-13 11:30:00+00:00,2026-01-15 18:00:00+00:00,9492,54.500000
5,2026-01-13 15:30:00+00:00,2026-01-15 18:00:00+00:00,9143,50.500000
6,2026-01-13 18:30:00+00:00,2026-01-15 18:00:00+00:00,9003,47.500000
7,2026-01-13 22:30:00+00:00,2026-01-15 18:00:00+00:00,8988,43.500000
8,2026-01-14 02:30:00+00:00,2026-01-15 18:00:00+00:00,8962,39.500000
9,2026-01-14 04:30:00+00:00,2026-01-15 18:00:00+00:00,8962,37.500000


### Applying the rule by hand (slot 2026-01-15 18:00, cutoff 2026-01-14 11:00 UTC)

- **Decision forecast:** row 11, published 2026-01-14 09:30, **9,053 MW**. It's the latest publish at or before 11:00.
- **Row 12** (11:30) comes 30 minutes after the cutoff, so it can never be picked.
- **If the gate were 09:20 instead:** row 11 (09:30) would also be too late, and the pick would move back to **row 10** (07:30, **8,962 MW**), a difference of 91 MW. The forecasts at 02:30, 04:30 and 07:30 are all 8,962, so moving the gate earlier within that range wouldn't change the value. A rule based on timestamps handles this automatically.


## Cell 9 – `select_decision_forecast`

In [13]:
def select_decision_forecast(forecasts, gate_time="11:00", days_before=1):
    """For each startTime, pick the latest forecast published at or before the decision cutoff.

    Cutoff = gate_time UK local time, days_before days before the startTime's UK-local delivery date,
    converted to UTC. E.g. a 2026-01-15 18:00 UTC slot -> cutoff 2026-01-14 11:00 GMT = 11:00 UTC;
    a 2025-07-15 18:00 UTC slot -> cutoff 2025-07-14 11:00 BST = 10:00 UTC.
    startTimes with no forecast published before their cutoff are dropped.
    """
    df = forecasts.copy()
    # Build the cutoff on naive UK wall-clock time, then attach the zone: this keeps "11:00"
    # meaning 11:00 local on both sides of a clock change (11:00 is never ambiguous or skipped).
    local_date = df["startTime"].dt.tz_convert("Europe/London").dt.tz_localize(None).dt.normalize()
    cutoff_local = local_date - pd.Timedelta(days=days_before) + pd.Timedelta(gate_time + ":00")
    df["cutoff"] = cutoff_local.dt.tz_localize("Europe/London").dt.tz_convert("UTC")

    eligible = df[df["publishTime"] <= df["cutoff"]]
    # sort + drop_duplicates instead of idxmax + .loc: it never looks at the index, so a
    # non-unique index (e.g. from pd.concat of chunks) can't silently produce extra rows.
    return (
        eligible.sort_values(["startTime", "publishTime"])
        .drop_duplicates(subset="startTime", keep="last")
        .reset_index(drop=True)
    )

In [14]:
decision = select_decision_forecast(jan_forecasts)
print(decision[decision["startTime"] == slot].to_string())
print(select_decision_forecast(jan_forecasts, gate_time="09:20").query("startTime == @slot").to_string())

# Index-safety: a non-unique index (as pd.concat of chunks can give) must not change the result
dup_index = jan_forecasts.copy()
dup_index.index = dup_index.index % 5000  # force label collisions
print("\nsame result with non-unique index:", select_decision_forecast(dup_index).equals(decision))

# DST: same rule in summer gives a 10:00 UTC cutoff
summer = pd.DataFrame({"publishTime": [pd.Timestamp("2025-07-14 09:30", tz="UTC")],
                       "startTime": [pd.Timestamp("2025-07-15 18:00", tz="UTC")], "generation": [0]})
print("summer cutoff:", select_decision_forecast(summer)["cutoff"].iloc[0])

                  publishTime                 startTime  generation                    cutoff
426 2026-01-14 09:30:00+00:00 2026-01-15 18:00:00+00:00        9053 2026-01-14 11:00:00+00:00
                  publishTime                 startTime  generation                    cutoff
426 2026-01-14 07:30:00+00:00 2026-01-15 18:00:00+00:00        8962 2026-01-14 09:20:00+00:00

same result with non-unique index: True
summer cutoff: 2025-07-14 10:00:00+00:00


## Scope: decisions made in January 2026

With a day-ahead gate (D−1 11:00 UK time), decisions made on **1–31 Jan** are for deliveries on **2 Jan to 1 Feb**. That means delivery `startTime` from 2026-01-02 00:00 UTC up to but not including 2026-02-02 00:00 UTC (31 × 24 = 744 hours). This scope is used everywhere from here on.


In [15]:
SCOPE_START = pd.Timestamp("2026-01-02 00:00", tz="UTC")
SCOPE_END = pd.Timestamp("2026-02-02 00:00", tz="UTC")  # exclusive

scoped = decision[(decision["startTime"] >= SCOPE_START) & (decision["startTime"] < SCOPE_END)]
print("slots with a decision forecast:", len(scoped), "(expected 744)")
print("one row per startTime:", scoped["startTime"].is_unique)
print("any publishTime > cutoff:", (scoped["publishTime"] > scoped["cutoff"]).any())
print("first decision cutoff:", scoped["cutoff"].min(), "| last:", scoped["cutoff"].max())

slots with a decision forecast: 744 (expected 744)
one row per startTime: True
any publishTime > cutoff: False
first decision cutoff: 2026-01-01 11:00:00+00:00 | last: 2026-01-31 11:00:00+00:00


## Cell 10 – Join forecasts to actuals

**Choice: A (average the two half-hour actuals into an hourly actual).** `generation` is average power (MW) over its period, so the mean of the :00 and :30 half-hours is exactly the average MW over the hour the forecast describes. That compares like with like, and it uses every actual. C would throw away half the actuals. B would compare at half-hour resolution a forecast that has no half-hour information.

Assumption: a WINDFOR `startTime` of 18:00 describes the hour 18:00–19:00.


In [16]:
# Settlement dates 2026-01-02..2026-02-01 = UTC 2026-01-02 00:00..2026-02-01 23:30 in winter (GMT)
jan_actuals = fetch_actuals("2026-01-02", "2026-02-01")
print("half-hourly actuals:", len(jan_actuals), "(expected 31 x 48 = 1488)")

hourly_actuals = (
    jan_actuals.assign(hour=jan_actuals["startTime"].dt.floor("h"))
    .groupby("hour")
    .agg(actual=("generation", "mean"), n_halves=("generation", "size"))
    .reset_index()
    .rename(columns={"hour": "startTime"})
)
print("hours missing a half-hour:", (hourly_actuals["n_halves"] != 2).sum())

joined = scoped[["startTime", "publishTime", "generation"]].rename(columns={"generation": "forecast"}).merge(
    hourly_actuals[["startTime", "actual"]], on="startTime", how="inner", validate="one_to_one"
)
print("joined hours:", len(joined), "(expected 744)")

joined["error"] = joined["forecast"] - joined["actual"]  # positive = over-forecast
mae = joined["error"].abs().mean()
bias = joined["error"].mean()
print(f"MAE:  {mae:,.0f} MW")
print(f"bias: {bias:+,.0f} MW  (forecast - actual)")
print(f"mean actual: {joined['actual'].mean():,.0f} MW")

half-hourly actuals: 1488 (expected 31 x 48 = 1488)
hours missing a half-hour: 0
joined hours: 744 (expected 744)
MAE:  2,013 MW
bias: +1,730 MW  (forecast - actual)
mean actual: 11,284 MW


## Cell 11 – Is the bias an alignment artefact?

Docs check: the Elexon API docs don't say whether a WINDFOR value is an hourly average or a value at that instant. The `/forecast/generation/wind/history` endpoint tags each row with a settlement period, and `startTime` 21:00 maps to **SP 43 (21:00–21:30)**. So `startTime` is the *start* of the labelled period. Hour-beginning is the reading most consistent with this, but it isn't proven.

Test: recompute the error with the half-hourly actuals shifted by ±30 min before averaging to hours. If the bias barely moves, the join alignment isn't the cause.


In [17]:
def hourly_error(shift_minutes):
    shifted = jan_actuals.assign(startTime=jan_actuals["startTime"] + pd.Timedelta(minutes=shift_minutes))
    hourly = shifted.groupby(shifted["startTime"].dt.floor("h"))["generation"].agg(["mean", "size"])
    hourly = hourly[hourly["size"] == 2]["mean"].rename("actual")  # drop edge hours with one half
    m = scoped.set_index("startTime")[["generation"]].join(hourly, how="inner")
    err = m["generation"] - m["actual"]
    return {"shift_min": shift_minutes, "hours": len(m), "MAE": round(err.abs().mean()), "bias": round(err.mean())}

# -30: hour H uses H:30 and (H+1):00 actuals;  +30: hour H uses (H-1):30 and H:00 (centred on H)
pd.DataFrame([hourly_error(s) for s in (-30, 0, 30)])

,shift_min,hours,MAE,bias
0,-30,743,2060,1739
1,0,744,2013,1730
2,30,743,1973,1721
